# Covariance Protocol History Storage And Load Audit

This local CPU notebook reads the saved `covariance_protocol_histories` campaign, validates that the history shards were produced by `classA_U1FGTN_gpu.run_markov_circuit(...)`, and reports storage plus local load cost without materializing the full arrays in memory. All shard inspection uses `np.load(..., mmap_mode='r')`.


## Source Campaign And Outputs

Default source pointer:
`colab_small_system_testing/gpu_data/covariance_protocol_histories/latest_campaign.json`

Expected fixed campaign:
- geometries `N16x20`, `N16x25`, and `N16x30`
- `nshell=1`, `cycles=10`, `init_mode='default'`, `dw_truncation=True`, `trial_orbitals='X'`
- `G_history=True`, `save_init=False`, `save_history_stride=None`, so each shard stores exactly 10 covariance matrices per sample
- `perfect_correction` with 10 saved samples per geometry
- `postselect` with one canonical deterministic saved sample per geometry after the driver override

Outputs written by this notebook:
`colab_small_system_testing/analysis_outputs/covariance_protocol_histories_cpu/<campaign_id>/`


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)

BUNDLE_NAME = 'colab_small_system_testing'
SOURCE_CAMPAIGN = 'covariance_protocol_histories'
ANALYSIS_NAME = 'covariance_protocol_histories_cpu'
CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
CAMPAIGN_MANIFEST_OVERRIDE = None
EXPECTED_PROTOCOLS = {'perfect_correction', 'postselect'}
EXPECTED_CONFIGS = {(16, 20, 1), (16, 25, 1), (16, 30, 1)}
EXPECTED_CYCLES = 10
EXPECTED_HISTORY_LENGTH = 10
EXPECTED_REQUESTED_SAMPLES = 10
EXPECTED_EFFECTIVE_SAMPLES = {'perfect_correction': 10, 'postselect': 1}

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'gpu_data').exists() and (candidate / 'notebooks').exists():
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME}')

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
LATEST_POINTER_PATH = GPU_DATA_ROOT / SOURCE_CAMPAIGN / 'latest_campaign.json'

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] gpu data root: {GPU_DATA_ROOT}')
print(f'[setup] latest pointer: {LATEST_POINTER_PATH}')


In [ ]:
def load_json(path: Path):
    with Path(path).open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic(path: Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp.replace(path)


def rel_to_bundle(path: Path) -> str:
    return str(Path(path).resolve().relative_to(BUNDLE_ROOT.resolve()))


def rel_to_gpu_data(path: Path) -> str:
    return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))


def gib(byte_count: int | float) -> float:
    return float(byte_count) / 1024 ** 3


def mib(byte_count: int | float) -> float:
    return float(byte_count) / 1024 ** 2


def resolve_campaign_manifest_path() -> Path:
    if CAMPAIGN_MANIFEST_OVERRIDE is not None:
        candidate = Path(CAMPAIGN_MANIFEST_OVERRIDE)
        if not candidate.is_absolute():
            candidate = (BUNDLE_ROOT / candidate).resolve()
        return candidate

    latest = load_json(LATEST_POINTER_PATH)
    relative = latest.get('campaign_manifest_path_relative') or latest.get('manifest')
    if relative is None:
        raise KeyError(f'{LATEST_POINTER_PATH} is missing campaign manifest path metadata')
    return (GPU_DATA_ROOT / relative).resolve()


def matrix_payload_bytes(nlayer: int, dtype: np.dtype) -> int:
    return int(nlayer) * int(nlayer) * int(np.dtype(dtype).itemsize)


def expected_raw_bytes(shape: tuple[int, ...], dtype: np.dtype) -> int:
    return int(np.prod(shape, dtype=np.int64)) * int(np.dtype(dtype).itemsize)


def expected_run_meta(protocol: str) -> dict:
    if protocol not in EXPECTED_PROTOCOLS:
        raise ValueError(f'Unexpected protocol: {protocol}')
    return {
        'saved_samples': EXPECTED_EFFECTIVE_SAMPLES[protocol],
        'postselect': protocol == 'postselect',
        'perfect_correction': protocol == 'perfect_correction',
    }


In [ ]:
CAMPAIGN_MANIFEST_PATH = resolve_campaign_manifest_path()
CAMPAIGN = load_json(CAMPAIGN_MANIFEST_PATH)

if CAMPAIGN.get('canonical_dynamics_entry_point') != CANONICAL_ENTRY_POINT:
    raise ValueError(f'Unexpected campaign entry point in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('campaign_name') != SOURCE_CAMPAIGN:
    raise ValueError(f'Unexpected campaign name in {CAMPAIGN_MANIFEST_PATH}')
if int(CAMPAIGN.get('cycles', -1)) != EXPECTED_CYCLES:
    raise ValueError(f'Unexpected cycle count in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('store_mode') != 'history':
    raise ValueError(f'Unexpected store_mode in {CAMPAIGN_MANIFEST_PATH}')
if int(CAMPAIGN.get('history_length_saved', -1)) != EXPECTED_HISTORY_LENGTH:
    raise ValueError(f'Unexpected history length in {CAMPAIGN_MANIFEST_PATH}')
if bool(CAMPAIGN.get('save_init')) is not False:
    raise ValueError(f'Unexpected save_init in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('save_history_stride') is not None:
    raise ValueError(f'Unexpected save_history_stride in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('snapshot_cycles') is not None:
    raise ValueError(f'Expected snapshot_cycles=None in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('init_mode') != 'default':
    raise ValueError(f'Unexpected init mode in {CAMPAIGN_MANIFEST_PATH}')
if CAMPAIGN.get('dtype') != 'complex128':
    raise ValueError(f'Unexpected dtype in {CAMPAIGN_MANIFEST_PATH}')
if bool(CAMPAIGN.get('dw_truncation')) is not True:
    raise ValueError(f'Unexpected dw_truncation in {CAMPAIGN_MANIFEST_PATH}')
if set((int(rec['Nx']), int(rec['Ny']), int(rec['nshell'])) for rec in CAMPAIGN.get('results', [])) != EXPECTED_CONFIGS:
    raise ValueError(f'Unexpected geometry set in {CAMPAIGN_MANIFEST_PATH}')

CAMPAIGN_ID = str(CAMPAIGN['campaign_id'])
OUTPUT_ROOT = BUNDLE_ROOT / 'analysis_outputs' / ANALYSIS_NAME / CAMPAIGN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'[setup] campaign manifest: {CAMPAIGN_MANIFEST_PATH}')
print(f'[setup] campaign id: {CAMPAIGN_ID}')
print(f'[setup] output root: {OUTPUT_ROOT}')


In [ ]:
def inspect_run(entry: dict) -> dict:
    protocol = str(entry['protocol'])
    expected = expected_run_meta(protocol)
    key = (int(entry['Nx']), int(entry['Ny']), int(entry['nshell']))
    if key not in EXPECTED_CONFIGS:
        raise ValueError(f'{protocol}: unexpected geometry {key}')

    nx, ny, nshell = key
    run_dir = GPU_DATA_ROOT / entry['run_dir_relative']
    manifest_path = GPU_DATA_ROOT / entry['manifest_path_relative']
    summary_rel = entry.get('summary_path_relative') or entry.get('run_summary_path_relative')
    summary_path = GPU_DATA_ROOT / summary_rel

    manifest = load_json(manifest_path)
    summary = load_json(summary_path)
    cfg = manifest.get('config', {})

    for source_name, payload in [('campaign entry', entry), ('run summary', summary)]:
        if (int(payload.get('Nx', -1)), int(payload.get('Ny', -1)), int(payload.get('nshell', -1))) != key:
            raise ValueError(f'{key} {protocol}: unexpected geometry in {source_name}')
        if int(payload.get('cycles', -1)) != EXPECTED_CYCLES:
            raise ValueError(f'{key} {protocol}: unexpected cycle count in {source_name}')
        if payload.get('snapshot_cycles') is not None:
            raise ValueError(f'{key} {protocol}: expected snapshot_cycles=None in {source_name}')
        if int(payload.get('history_length_saved', -1)) != EXPECTED_HISTORY_LENGTH:
            raise ValueError(f'{key} {protocol}: unexpected history length in {source_name}')

    if int(summary.get('samples_requested', -1)) != EXPECTED_REQUESTED_SAMPLES:
        raise ValueError(f'{key} {protocol}: unexpected requested sample count')
    if int(summary.get('samples', -1)) != int(expected['saved_samples']):
        raise ValueError(f'{key} {protocol}: unexpected effective sample count')
    if bool(summary.get('postselect')) is not bool(expected['postselect']):
        raise ValueError(f'{key} {protocol}: postselect flag mismatch')
    if bool(summary.get('perfect_correction')) is not bool(expected['perfect_correction']):
        raise ValueError(f'{key} {protocol}: perfect_correction flag mismatch')
    if summary.get('canonical_dynamics_entry_point') != CANONICAL_ENTRY_POINT:
        raise ValueError(f'{key} {protocol}: unexpected canonical entry point in summary')

    if manifest.get('store_mode') != 'history' or cfg.get('store_mode') != 'history':
        raise ValueError(f'{key} {protocol}: expected history store mode')
    if cfg.get('sequence') != 'raster_y':
        raise ValueError(f'{key} {protocol}: expected raster_y sequence')
    if cfg.get('dtype') != 'c128':
        raise ValueError(f'{key} {protocol}: expected c128 manifest dtype')
    if cfg.get('init_mode') != 'default':
        raise ValueError(f'{key} {protocol}: expected default init mode')
    if bool(cfg.get('postselect')) is not bool(expected['postselect']):
        raise ValueError(f'{key} {protocol}: manifest postselect flag mismatch')
    if bool(cfg.get('perfect_correction')) is not bool(expected['perfect_correction']):
        raise ValueError(f'{key} {protocol}: manifest perfect_correction flag mismatch')
    if bool(cfg.get('DW')) is not True or bool(cfg.get('dw_truncation')) is not True:
        raise ValueError(f'{key} {protocol}: expected DW=True and dw_truncation=True')
    if cfg.get('trial_orbitals') != 'X':
        raise ValueError(f'{key} {protocol}: expected trial orbitals X')
    if cfg.get('snapshot_cycles') is not None:
        raise ValueError(f'{key} {protocol}: expected manifest snapshot_cycles=None')
    if bool(cfg.get('save_init')) is not False:
        raise ValueError(f'{key} {protocol}: expected manifest save_init=False')
    if cfg.get('save_history_stride') is not None:
        raise ValueError(f'{key} {protocol}: expected manifest save_history_stride=None')

    shards = manifest.get('shards', {})
    if sorted(int(k) for k in shards) != [0]:
        raise ValueError(f'{key} {protocol}: expected exactly one shard')
    shard_meta = shards['0']
    shard_path = run_dir / shard_meta['filename']
    if not shard_path.exists():
        raise FileNotFoundError(shard_path)

    shard = np.load(shard_path, mmap_mode='r', allow_pickle=False)
    shape = tuple(int(x) for x in shard.shape)
    dtype = np.dtype(shard.dtype)
    nlayer = 2 * nx * ny
    expected_shape = (int(expected['saved_samples']), EXPECTED_HISTORY_LENGTH, nlayer, nlayer)
    if shape != expected_shape:
        raise ValueError(f'{key} {protocol}: expected shard shape {expected_shape}, got {shape}')
    if dtype != np.dtype(np.complex128):
        raise ValueError(f'{key} {protocol}: expected complex128 shard, got {dtype}')
    if [int(x) for x in shard_meta.get('shape', [])] != list(expected_shape):
        raise ValueError(f'{key} {protocol}: manifest shard shape mismatch')

    raw_payload_bytes = expected_raw_bytes(shape, dtype)
    file_bytes = shard_path.stat().st_size
    header_bytes = int(file_bytes - raw_payload_bytes)
    if header_bytes < 0:
        raise ValueError(f'{key} {protocol}: shard file is smaller than raw payload')

    one_matrix_bytes = matrix_payload_bytes(nlayer, dtype)
    row = {
        'Nx': nx,
        'Ny': ny,
        'nshell': nshell,
        'protocol': protocol,
        'samples_requested': int(summary['samples_requested']),
        'samples_saved': int(summary['samples']),
        'history_length_saved': EXPECTED_HISTORY_LENGTH,
        'nlayer': nlayer,
        'dtype': str(dtype),
        'batch_size': int(summary['batch_size']),
        'batch_size_mode': summary['batch_size_mode'],
        'run_dir_relative': rel_to_gpu_data(run_dir),
        'manifest_path_relative': rel_to_gpu_data(manifest_path),
        'summary_path_relative': rel_to_gpu_data(summary_path),
        'shard_path_relative': rel_to_gpu_data(shard_path),
        'shard_shape': list(shape),
        'raw_payload_bytes': int(raw_payload_bytes),
        'raw_payload_gib': round(gib(raw_payload_bytes), 4),
        'file_size_bytes': int(file_bytes),
        'file_size_gib': round(gib(file_bytes), 4),
        'npy_header_overhead_bytes': header_bytes,
        'npy_header_overhead_kib': round(header_bytes / 1024.0, 3),
        'full_read_io_bytes': int(file_bytes),
        'full_read_io_gib': round(gib(file_bytes), 4),
        'peak_eager_load_bytes': int(raw_payload_bytes),
        'peak_eager_load_gib': round(gib(raw_payload_bytes), 4),
        'peak_memmap_stream_matrix_bytes': int(one_matrix_bytes),
        'peak_memmap_stream_matrix_mib': round(mib(one_matrix_bytes), 4),
        'dw_loc': summary.get('dw_loc'),
        'topological_x_range': summary.get('topological_x_range'),
        'postselect': bool(summary['postselect']),
        'perfect_correction': bool(summary['perfect_correction']),
    }
    del shard
    return row


run_rows = [inspect_run(entry) for entry in CAMPAIGN.get('results', [])]
run_rows = sorted(run_rows, key=lambda row: (row['Nx'], row['Ny'], row['protocol']))
per_run_df = pd.DataFrame(run_rows)

total_row = {
    'campaign_id': CAMPAIGN_ID,
    'run_count': int(len(per_run_df)),
    'total_saved_samples': int(per_run_df['samples_saved'].sum()),
    'total_history_matrices': int((per_run_df['samples_saved'] * per_run_df['history_length_saved']).sum()),
    'total_raw_payload_bytes': int(per_run_df['raw_payload_bytes'].sum()),
    'total_raw_payload_gib': round(gib(per_run_df['raw_payload_bytes'].sum()), 4),
    'total_file_size_bytes': int(per_run_df['file_size_bytes'].sum()),
    'total_file_size_gib': round(gib(per_run_df['file_size_bytes'].sum()), 4),
    'total_full_read_io_bytes': int(per_run_df['full_read_io_bytes'].sum()),
    'total_full_read_io_gib': round(gib(per_run_df['full_read_io_bytes'].sum()), 4),
    'largest_single_shard_gib': round(float(per_run_df['file_size_gib'].max()), 4),
    'largest_eager_load_gib': round(float(per_run_df['peak_eager_load_gib'].max()), 4),
    'streaming_one_matrix_mib': round(float(per_run_df['peak_memmap_stream_matrix_mib'].max()), 4),
}
totals_df = pd.DataFrame([total_row])

summary_csv_path = OUTPUT_ROOT / 'storage_load_summary.csv'
summary_json_path = OUTPUT_ROOT / 'storage_load_summary.json'
analysis_manifest_path = OUTPUT_ROOT / 'analysis_manifest.json'

per_run_df.to_csv(summary_csv_path, index=False)
write_json_atomic(
    summary_json_path,
    {
        'campaign_manifest': rel_to_bundle(CAMPAIGN_MANIFEST_PATH),
        'per_run': per_run_df.to_dict(orient='records'),
        'totals': total_row,
    },
)
write_json_atomic(
    analysis_manifest_path,
    {
        'analysis_name': ANALYSIS_NAME,
        'campaign_id': CAMPAIGN_ID,
        'source_campaign': SOURCE_CAMPAIGN,
        'source_manifest': rel_to_bundle(CAMPAIGN_MANIFEST_PATH),
        'generated_at': datetime.now().isoformat(timespec='seconds'),
        'outputs': [
            'storage_load_summary.csv',
            'storage_load_summary.json',
        ],
    },
)

print(f'[done] wrote {summary_csv_path}')
print(f'[done] wrote {summary_json_path}')
print(f'[done] wrote {analysis_manifest_path}')


In [ ]:
display(per_run_df)
display(totals_df)
